# SeqDance embeddings for IDR-GFP fusion constructs

Drop-in twin of `compute_esm_embeddings.ipynb`, but using **SeqDance** instead of ESM-2.

**Why SeqDance:** it is the ESM2-35M architecture *retrained from scratch on protein dynamics* (MD + NMA, including the IDRome CG-MD set). Its embeddings encode conformational properties (Rg, Re, RMSF) that ESM is largely blind to, so they should be far more sensitive to the charge effect you care about. (ESMDance just freezes ESM2 and adds heads -> its encoder embeddings == ESM2's, so it is useless as a new embedding.)

**Model**: `ChaoHou/SeqDance` weights loaded into the `ESMwrap` class from <https://github.com/ShenLab/SeqDance>. Backbone = `facebook/esm2_t12_35M_UR50D` -> **480-dim** embeddings (not 1280).

**Inputs**: auto-clones your repo for `constructs/all_constructs.fasta` and `subset/IDRome_subset_cdhit50.fasta`, and the SeqDance repo for the model code.

**Outputs** on Drive under `MyDrive/idrome_globular/embeddings/seqdance/`:
- `per_construct_in_context/<id>.pt`  -- per-residue (L,480) float16 + IDR/full mean-pool + metadata
- `incontext_meanpool.parquet`        -- one row per construct (`mean_full_*`, `mean_idr_*` + metadata)
- `idr_alone_meanpool.parquet` + `per_construct_idr_alone/<id>.pt` -- the 521 bare-IDR baselines

Resume-safe: re-run the main loop after any disconnect; it skips finished constructs.

## Before running
1. **Runtime -> Change runtime type -> GPU** (T4 is plenty; SeqDance is only 35M params -> much faster than ESM-650M).
2. Run cells top-to-bottom.

## 1. Install deps, mount Drive, clone both repos

In [ ]:
# SeqDance pins: transformers==4.48.2 (their env). huggingface_hub needed for PyTorchModelHubMixin.
!pip install -q transformers==4.48.2 huggingface_hub biopython pyarrow tqdm

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess

# (a) your repo -> FASTA inputs
REPO_URL = 'https://github.com/hamidrgh/idrome_globular.git'
REPO_DIR = '/content/idrome_globular'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

# (b) SeqDance repo -> model code (model/model.py, model/config.py)
SEQDANCE_URL = 'https://github.com/ShenLab/SeqDance.git'
SEQDANCE_DIR = '/content/SeqDance'
if not os.path.isdir(SEQDANCE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', SEQDANCE_URL, SEQDANCE_DIR], check=True)

!ls {REPO_DIR}/constructs | head -3
!ls {SEQDANCE_DIR}/model

## 2. Configuration

In [ ]:
import os

# inputs (read-only) -- from the cloned GitHub repo
FASTA_PATH        = f'{REPO_DIR}/constructs/all_constructs.fasta'
IDR_FASTA         = f'{REPO_DIR}/subset/IDRome_subset_cdhit50.fasta'

# outputs -- persisted to Drive, in a dedicated seqdance/ folder so they never
# collide with the ESM outputs (same filenames, so the Tier scripts just point
# at a different folder + use HIDDEN_DIM=480)
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/idrome_globular'
OUT_DIR           = f'{DRIVE_PROJECT_DIR}/embeddings/seqdance'
PER_CONSTRUCT_DIR = f'{OUT_DIR}/per_construct_in_context'
SUMMARY_PATH      = f'{OUT_DIR}/incontext_meanpool.parquet'
IDR_ALONE_PARQ    = f'{OUT_DIR}/idr_alone_meanpool.parquet'
PER_IDR_DIR       = f'{OUT_DIR}/per_construct_idr_alone'

# SeqDance model selection
ESM2_SELECT       = 'model_35M'      # SeqDance backbone (only option)
MODEL_SELECT      = 'seqdance'       # 'seqdance' (use this) or 'esmdance' (== ESM2-35M embeddings)
HF_WEIGHTS        = 'ChaoHou/SeqDance'
TOKENIZER_NAME    = 'facebook/esm2_t12_35M_UR50D'

MAX_TOKENS        = 1022            # 1024 - 2 (CLS, EOS)
CHUNK_OVERLAP     = 200             # sliding window on long sequences
SAVE_FLOAT16      = True
CHECKPOINT_EVERY  = 100

os.makedirs(PER_CONSTRUCT_DIR, exist_ok=True)
os.makedirs(PER_IDR_DIR, exist_ok=True)
print('fasta in :', FASTA_PATH, '(exists)' if os.path.exists(FASTA_PATH) else '(MISSING!)')
print('out dir  :', OUT_DIR)
print('model    :', HF_WEIGHTS, '/', MODEL_SELECT)

## 3. Load FASTA and parse metadata from headers

Identical convention to the ESM notebook: first header token = `construct_id`, `| key=value` pairs = metadata.

In [ ]:
from Bio import SeqIO

records = []
for rec in SeqIO.parse(FASTA_PATH, 'fasta'):
    meta = {}
    if rec.description and '|' in rec.description:
        for piece in rec.description.split('|')[1:]:
            piece = piece.strip()
            if '=' in piece:
                k, v = piece.split('=', 1)
                meta[k.strip()] = v.strip()
    records.append({'construct_id': rec.id, 'sequence': str(rec.seq), **meta})

lengths = [len(r['sequence']) for r in records]
n_long  = sum(1 for L in lengths if L > MAX_TOKENS)
print(f'loaded {len(records)} constructs')
print(f'length range : {min(lengths)} - {max(lengths)}')
print(f'long (>{MAX_TOKENS} aa) : {n_long} (sliding window will be used)')

## 4. Load SeqDance

We instantiate `ESMwrap` (their wrapper around `EsmModel` + dynamic-property heads) and load the trained weights from HuggingFace. For embeddings we only need the trained encoder `dance_model.esm2`, so we call it directly (skips the attention-map computation in `ESMwrap.forward` -> faster, less memory). The `last_hidden_state` of this encoder *is* the SeqDance embedding.

In [ ]:
import sys, torch
from transformers import AutoTokenizer

sys.path.insert(0, f'{SEQDANCE_DIR}/model')   # so `from config import config` inside model.py resolves
from model import ESMwrap

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device :', DEVICE)
if DEVICE == 'cuda':
    print('GPU    :', torch.cuda.get_device_name())

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

# build then load trained weights (PyTorchModelHubMixin restores the ctor kwargs)
dance_model = ESMwrap(ESM2_SELECT, MODEL_SELECT)
dance_model = dance_model.from_pretrained(HF_WEIGHTS)
dance_model = dance_model.to(DEVICE).eval()

# the trained encoder; its last_hidden_state is the SeqDance embedding
encoder = dance_model.esm2
HIDDEN_DIM = encoder.config.hidden_size
print('hidden dim :', HIDDEN_DIM, '(expected 480)')

## 5. Embedding function (single-sequence, sliding window for long seqs)

Same strip-CLS/EOS + mean-pool convention as the ESM notebook, so outputs are positionally comparable.

In [ ]:
@torch.inference_mode()
def embed_sequence(seq: str,
                   max_len: int = MAX_TOKENS,
                   overlap: int = CHUNK_OVERLAP) -> torch.Tensor:
    """Return per-residue SeqDance embedding (L, D) on CPU as float32."""
    L = len(seq)
    D = HIDDEN_DIM

    if L <= max_len:
        toks = tokenizer(seq, return_tensors='pt').to(DEVICE)
        out  = encoder(**toks).last_hidden_state         # (1, L+2, D)
        return out[0, 1:-1].float().cpu()                # strip CLS/EOS -> (L, D)

    accum  = torch.zeros(L, D, dtype=torch.float32)
    weight = torch.zeros(L,    dtype=torch.float32)
    step   = max_len - overlap
    starts = list(range(0, L, step))
    if starts[-1] + max_len < L:
        starts.append(L - max_len)
    for start in starts:
        end   = min(start + max_len, L)
        chunk = seq[start:end]
        toks  = tokenizer(chunk, return_tensors='pt').to(DEVICE)
        out   = encoder(**toks).last_hidden_state
        emb   = out[0, 1:-1].float().cpu()
        accum[start:end]  += emb
        weight[start:end] += 1.0
    return accum / weight.unsqueeze(1)

# quick smoke test
_t = embed_sequence('MKKAEEGGSDEEKRKRPADDDEE')
print('test emb shape:', tuple(_t.shape))

## 6. Main loop -- compute & save construct embeddings

Resume-safe; flushes the mean-pool summary every `CHECKPOINT_EVERY` constructs with an atomic write.

In [ ]:
import time
import pandas as pd
from tqdm.auto import tqdm

META_KEYS = ['idr', 'gfp', 'topology', 'N_idr', 'N_construct',
             'ncpr_idr', 'fcr_idr', 'kappa_idr', 'nu_idr_free', 'Z_gfp']

def _maybe_num(s: str):
    try:    return float(s)
    except: return s

summary_buffer: list[dict] = []

def flush_summary():
    if not summary_buffer:
        return
    df = pd.DataFrame(summary_buffer)
    if os.path.exists(SUMMARY_PATH):
        try:
            old = pd.read_parquet(SUMMARY_PATH)
            df = (pd.concat([old, df], ignore_index=True)
                    .drop_duplicates('construct_id', keep='last'))
        except Exception as e:
            print(f'warn: could not read existing summary ({e}); overwriting with buffer')
    tmp = SUMMARY_PATH + '.tmp'
    df.to_parquet(tmp, index=False)
    os.replace(tmp, SUMMARY_PATH)
    summary_buffer.clear()

records_sorted = sorted(records, key=lambda r: len(r['sequence']))
t0 = time.time()
n_skipped = 0

for rec in tqdm(records_sorted):
    cid = rec['construct_id']
    out_path = os.path.join(PER_CONSTRUCT_DIR, f'{cid}.pt')
    if os.path.exists(out_path):
        n_skipped += 1
        continue

    emb = embed_sequence(rec['sequence'])            # (L, D) float32 CPU

    topo  = rec.get('topology', '')
    n_idr = int(float(rec.get('N_idr', 0)))
    if topo == 'Ntail':
        idr_emb = emb[:n_idr]
    elif topo == 'Ctail':
        idr_emb = emb[-n_idr:]
    else:
        idr_emb = emb

    mean_full = emb.mean(dim=0)
    mean_idr  = idr_emb.mean(dim=0)

    payload = {
        'construct_id'  : cid,
        'sequence'      : rec['sequence'],
        'residue_emb'   : emb.to(torch.float16) if SAVE_FLOAT16 else emb,
        'mean_pool_full': mean_full,
        'mean_pool_idr' : mean_idr,
        'metadata'      : {k: _maybe_num(rec.get(k, '')) for k in META_KEYS},
    }
    torch.save(payload, out_path)

    row = {'construct_id': cid}
    for k in META_KEYS:
        row[k] = _maybe_num(rec.get(k, ''))
    row.update({f'mean_full_{i}': float(mean_full[i]) for i in range(HIDDEN_DIM)})
    row.update({f'mean_idr_{i}' : float(mean_idr[i])  for i in range(HIDDEN_DIM)})
    summary_buffer.append(row)

    if len(summary_buffer) >= CHECKPOINT_EVERY:
        flush_summary()

flush_summary()
print(f'\ndone in {(time.time()-t0)/60:.1f} min')
print(f'  newly embedded : {len(records_sorted) - n_skipped}')
print(f'  skipped (cached): {n_skipped}')

## 7. Sanity check

In [ ]:
import pandas as pd, torch, os

summary = pd.read_parquet(SUMMARY_PATH)
print('summary shape    :', summary.shape)
print('unique constructs:', summary['construct_id'].nunique())
print('hidden dim cols  :', sum(c.startswith('mean_idr_') for c in summary.columns))

cid = summary['construct_id'].iloc[0]
p = torch.load(os.path.join(PER_CONSTRUCT_DIR, f'{cid}.pt'), map_location='cpu')
print(f'\n--- {cid} ---')
for k, v in p.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k:18}  shape={tuple(v.shape)}  dtype={v.dtype}')
    elif isinstance(v, dict):
        print(f'  {k:18}  {v}')
    else:
        print(f'  {k:18}  {type(v).__name__}')

## 8. Isolated-IDR baseline (E_alone)

Embeds the 521 bare IDR sequences with the same model/convention, giving the `E_alone` baseline directly comparable to the construct `mean_idr_*` columns (`E_context`). Writes both a parquet (`mean_0 .. mean_479`) and per-residue `.pt` files (needed for the position-resolved Tier-5 analysis).

In [ ]:
import os, glob, torch, pandas as pd
from Bio import SeqIO
from tqdm.auto import tqdm

idr_records = [(rec.id, str(rec.seq)) for rec in SeqIO.parse(IDR_FASTA, 'fasta')]
print(f'isolated IDRs to embed: {len(idr_records)}')

done = set()
if os.path.exists(IDR_ALONE_PARQ):
    try:
        done = set(pd.read_parquet(IDR_ALONE_PARQ, columns=['seq_name'])['seq_name'])
        print(f'already in parquet: {len(done)}')
    except Exception as e:
        print(f'existing parquet unreadable ({e}); recomputing all')

rows = []
for seq_name, seq in tqdm(sorted(idr_records, key=lambda r: len(r[1]))):
    pt_path = os.path.join(PER_IDR_DIR, f'{seq_name}.pt')
    if seq_name in done and os.path.exists(pt_path):
        continue

    emb  = embed_sequence(seq)            # (L, D); whole seq = IDR
    mean = emb.mean(dim=0)

    if not os.path.exists(pt_path):
        torch.save({
            'seq_name'      : seq_name,
            'sequence'      : seq,
            'residue_emb'   : emb.to(torch.float16) if SAVE_FLOAT16 else emb,
            'mean_pool_idr' : mean,
        }, pt_path)

    if seq_name not in done:
        row = {'seq_name': seq_name, 'N_idr': len(seq)}
        row.update({f'mean_{i}': float(mean[i]) for i in range(HIDDEN_DIM)})
        rows.append(row)

df_new = pd.DataFrame(rows)
if rows:
    if os.path.exists(IDR_ALONE_PARQ):
        try:
            df_old = pd.read_parquet(IDR_ALONE_PARQ)
            df_new = (pd.concat([df_old, df_new], ignore_index=True)
                        .drop_duplicates('seq_name', keep='last'))
        except Exception:
            pass
    tmp = IDR_ALONE_PARQ + '.tmp'
    df_new.to_parquet(tmp, index=False)
    os.replace(tmp, IDR_ALONE_PARQ)

final = pd.read_parquet(IDR_ALONE_PARQ)
n_pt  = len(glob.glob(os.path.join(PER_IDR_DIR, '*.pt')))
print(f'wrote {IDR_ALONE_PARQ}')
print(f'  parquet rows         : {len(final)}   (expected {len(idr_records)})')
print(f'  per-residue .pt files: {n_pt}   (expected {len(idr_records)})')